In [5]:
import pandas as pd
  
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [7]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Pregunta A (Sumarización Categórica): Usen la función .value_counts(normalize=True) en la columna Survived (0 = Murió, 1 = Sobrevivió). ¿Cuál es la tasa de supervivencia global del barco expresada en porcentaje?

Murio = 61.61%
Sobrevivio = 100% - 61.61% looool

In [8]:
df['Survived'].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Pregunta B (Agrupación y Agregación): El famoso código marítimo era "mujeres y niños primero". Comprobemos esto matemáticamente. Ejecuten una agrupación por la columna de género y calculen el promedio de la columna Survived. ¿Qué porcentaje exacto de mujeres sobrevivió en contraste con los hombres?

Mujeres = 74.20%
Hombres = 18.89%

In [9]:
df.groupby('Sex')['Survived'].mean(numeric_only=True)

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

Pregunta C: El director financiero nota que la tarifa máxima (Fare) cobrada fue de más de 500 libras, mientras que el promedio ronda las 32. Sospecha de outliers. Utilicen las herramientas de dispersión en Pandas para confirmarlo:
Calculen el Cuartil 1 (25%) y el Cuartil 3 (75%) de la columna Fare usando el método .quantile([0.25, 0.75]).
Calculen matemáticamente el Rango Intercuartílico (IQR = Q3 - Q1).
Calculen el límite superior aceptable (Q3 + 1.5 * IQR). Escriban una línea de código para filtrar el DataFrame y descubrir exactamente cuántos pasajeros pagaron una tarifa por encima de ese límite matemático. ¿A qué clase (Pclass) pertenecían la mayoría de ellos?

116 omg



In [13]:
iqr = df['Fare'].quantile(.75) - df['Fare'].quantile(.25)


maximum = df['Fare'].quantile(.75) + 1.5*iqr
allMax = df[df['Fare'] > maximum].shape[0]

allMax



116

Pregunta D: Calculen la media y la mediana de la columna Fare. Notarán una diferencia enorme entre ambos valores. Matemáticamente, ¿qué significa que la media sea tan superior a la mediana? Si en el futuro utilizamos un algoritmo basado en distancias euclidianas (como K-Nearest Neighbors) sin escalar previamente esta variable, ¿cómo afectará esta asimetría al aprendizaje del modelo?


Esto significa que hay mucha varianza como visto antes en la prueba de cuartiles hacia valores maximos, esto haria que los casos en los que el Fare tiene un valor menor no afecten tanto en inferencias y este sesgado a Fares mayores

In [16]:
media = df['Fare'].mean()
mediana = df['Fare'].median()

print(f"media: {media}")
print(f"mediana: {mediana}")


media: 32.204207968574636
mediana: 14.4542


Pregunta E: En la semana 2 vimos que el muestreo aleatorio simple es peligroso. El objetivo es entrenar un modelo que prediga la supervivencia (Survived), pero las clases están desbalanceadas. Escriban el código en Pandas para extraer una muestra de exactamente 150 pasajeros garantizando que la proporción de sobrevivientes y no sobrevivientes en la muestra sea idéntica a la de la base de datos completa. ¿Qué sesgo evitan al hacer esto?

Evitamos que el peso de cierto valor sea identico para que no haya bias en las pruebas

In [ ]:
proporciones = df['Survived'].value_counts(normalize=True)

muestra = (proporciones * 150).round().astype(int)


muestra_estratificada = df.groupby('Survived', group_keys=False).apply(
    lambda x: x.sample(n=muestra[x.name], random_state=42)
)

muestra_estratificada['Survived'].value_counts(normalize=True)
muestra_estratificada['Survived'].count()




Proporción de sobrevivientes en la muestra estratificada: Survived
0    0.613333
1    0.386667
Name: proportion, dtype: float64      


C:\Users\XxDus\AppData\Local\Temp\ipykernel_18296\3051849473.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  muestra_estratificada = df.groupby('Survived', group_keys=False).apply(


Si ejecutan df.groupby('Survived')['Age'].mean(), Pandas calculará el promedio de edad de los que vivieron y los que murieron. Sin embargo, por defecto, Pandas ignora los valores NaN al calcular la media. Si resulta que la gran mayoría de las edades faltantes (NaN) pertenecían a pasajeros de 3ra clase que murieron, ¿qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?

Esto causaria un sesgo hacia que indica que arruina el ratio de muertes, por lo que parece que la gente que murio/vivio es parejo cuando en realidad se ignoran muchas muertes, esto afectaria nuestro modelo de prediccion causando mas posibles falsos positivos (sobrevivientes) 

In [56]:
df.groupby('Survived')['Age'].mean()

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

Con el cálculo del IQR en la Pregunta C, determinaron que los boletos de 512 libras son outliers matemáticos. En Machine Learning, los valores atípicos pueden representar errores de captura (ruido que aumenta el error irreducible) o casos especiales válidos (señal). Investigando la naturaleza de un barco de lujo, ¿deberíamos eliminar estas filas con .drop() antes de entrenar nuestro modelo? Justifiquen su respuesta arquitectónica.

No creo que sea buena idea removerlos, claramente nos va a perjudicar ya que aunque sean casos extremos, siguen siendo informacion real teniendo en cuenta que es un barco de lujo, de hecho uno podria pensar que si pagan mas podrian tener mas probabilidad de sobrevivir dado a que estan en mejores ubicaciones, por lo que eliminarlos seria perder datos valiosos. Entonces no, no deberíamos eliminarlos

Observen la columna Name. Es texto libre (dato no estructurado), pero contiene títulos ocultos como "Mr.", "Mrs.", "Miss." o "Master.". Si lograran extraer ese título usando expresiones regulares en Pandas, podrían hacer un .groupby('Titulo')['Age'].median(). ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?

Porque no estaríamos asumiendo datos y es una buena manera de feature engineering porque recuperamos datos reales que se habrían perdido o que simplemente hubieramos aplastado/biased usando la mediana  

Calculen la varianza de la columna Survived. Dado que es una variable categórica codificada como 0 y 1, el resultado numérico estará cerca de $0.23$. Matemáticamente, ¿qué significaría si la varianza de esta variable fuera exactamente 0.0? ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?

Esto significa que o todos murieron o todos sobrevivieron, entonces no habria mucho que entrenar al modelo y estaria super sesgado sacando falsos positivos/negativos    al probar un dataset con dataset normal

In [58]:
df['Survived'].var()

0.2367722165474984

Ejecuten una agrupación por tres niveles al mismo tiempo y cuenten cuántos pasajeros hay en cada subgrupo: df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count(). Notarán que algunos subgrupos tienen 1 o 2 pasajeros. Si un algoritmo intenta extraer reglas de probabilidad de grupos tan pequeños, se enfrentará a la "Maldición de la Dimensionalidad" (Curse of Dimensionality). ¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?

Tendria un sobre ajuste o overfitting debido a que es la unica referencia que tiene el modelo para ese caso, por lo que lo que sea que este labeled para ese dato, en ese especifico groupby o join, tendra esa misma respuesta para cualquier dato que comparta esas 3 clases en un nuevo datasetet con mas rows 

In [59]:
df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count()

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64